In [2]:
import pandas as pd
import numpy as np
import random
from pymongo import MongoClient

# Fetch directly from MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['ecommerce_db']
collection = db['criteo_raw']

df = pd.DataFrame(list(collection.find()))
df = df.drop(columns=['_id'])

print("Fetched from MongoDB:")
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Fetched from MongoDB:
Shape: (100000, 22)
Missing values: 0

Columns: ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id', 'attribution', 'click', 'click_pos', 'click_nb', 'cost', 'cpo', 'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']

First 3 rows:
   timestamp       uid  campaign  conversion  conversion_timestamp  \
0          0  20073966  22589171           0                    -1   
1          2  24607497    884761           0                    -1   
2          2  28474333  18975823           0                    -1   

   conversion_id  attribution  click  click_pos  click_nb  ...  \
0             -1            0      0         -1        -1  ...   
1             -1            0      0         -1        -1  ...   
2             -1            0      0         -1        -1  ...   

   time_since_last_click      cat1     cat2      cat3      cat4      cat5  \
0                     -1   5824233  9312274

In [4]:
np.random.seed(42)
random.seed(42)

df_noisy = df.copy()

# Convert campaign to string type first before any noise injection
df_noisy['campaign'] = df_noisy['campaign'].astype(str)

# Also convert timestamp to float so it can accept -1
df_noisy['timestamp'] = df_noisy['timestamp'].astype(float)
df_noisy['time_since_last_click'] = df_noisy['time_since_last_click'].astype(float)

total_rows = len(df_noisy)
print(f"Starting noise injection on {total_rows:,} rows...\n")

# Noise 1: Missing values in cost and cpo
missing_idx = np.random.choice(total_rows, size=int(total_rows * 0.05), replace=False)
df_noisy.loc[missing_idx, 'cost'] = np.nan
print(f"1. Added {len(missing_idx):,} missing values to 'cost' (5%)")

missing_idx2 = np.random.choice(total_rows, size=int(total_rows * 0.04), replace=False)
df_noisy.loc[missing_idx2, 'cpo'] = np.nan
print(f"2. Added {len(missing_idx2):,} missing values to 'cpo' (4%)")

# Noise 2: Sentinel -1 values in time_since_last_click
sentinel_idx = np.random.choice(total_rows, size=int(total_rows * 0.08), replace=False)
df_noisy.loc[sentinel_idx, 'time_since_last_click'] = -1
print(f"3. Added {len(sentinel_idx):,} sentinel -1 values to 'time_since_last_click' (8%)")

# Noise 3: Duplicate rows
dup_idx = np.random.choice(total_rows, size=int(total_rows * 0.02), replace=False)
duplicates = df_noisy.iloc[dup_idx].copy()
df_noisy = pd.concat([df_noisy, duplicates], ignore_index=True)
print(f"4. Added {len(dup_idx):,} duplicate rows (2%) — new shape: {df_noisy.shape}")

# Noise 4: Outlier cost values
outlier_idx = np.random.choice(total_rows, size=int(total_rows * 0.01), replace=False)
df_noisy.loc[outlier_idx, 'cost'] = df_noisy['cost'].median() * 1000
print(f"5. Added {len(outlier_idx):,} extreme outlier values to 'cost' (1%)")

# Noise 5: Inconsistent campaign formatting — now safe because campaign is string
format_idx = np.random.choice(total_rows, size=int(total_rows * 0.03), replace=False)
df_noisy.loc[format_idx, 'campaign'] = df_noisy.loc[format_idx, 'campaign'].apply(
    lambda x: f"CAMP_{x}" if random.random() > 0.5 else x.lower()
)
print(f"6. Added {len(format_idx):,} inconsistent campaign format values (3%)")

# Noise 6: Invalid timestamp values
ts_idx = np.random.choice(total_rows, size=int(total_rows * 0.02), replace=False)
df_noisy.loc[ts_idx, 'timestamp'] = -1
print(f"7. Added {len(ts_idx):,} invalid timestamp values (2%)")

# Noise 7: Missing campaign values
camp_idx = np.random.choice(total_rows, size=int(total_rows * 0.03), replace=False)
df_noisy.loc[camp_idx, 'campaign'] = np.nan
print(f"8. Added {len(camp_idx):,} missing campaign values (3%)")

print(f"\nNoise injection complete.")
print(f"Final noisy shape: {df_noisy.shape}")
print(f"\nMissing values summary:")
print(df_noisy.isnull().sum()[df_noisy.isnull().sum() > 0])
print(f"\nDuplicate rows: {df_noisy.duplicated().sum():,}")
print(f"Sentinel -1 in time_since_last_click: {(df_noisy['time_since_last_click'] == -1).sum():,}")

Starting noise injection on 100,000 rows...

1. Added 5,000 missing values to 'cost' (5%)
2. Added 4,000 missing values to 'cpo' (4%)
3. Added 8,000 sentinel -1 values to 'time_since_last_click' (8%)
4. Added 2,000 duplicate rows (2%) — new shape: (102000, 22)
5. Added 1,000 extreme outlier values to 'cost' (1%)
6. Added 3,000 inconsistent campaign format values (3%)
7. Added 2,000 invalid timestamp values (2%)
8. Added 3,000 missing campaign values (3%)

Noise injection complete.
Final noisy shape: (102000, 22)

Missing values summary:
campaign    3000
cost        5033
cpo         4081
dtype: int64

Duplicate rows: 1,857
Sentinel -1 in time_since_last_click: 55,250


In [5]:
#save noisy json

import os
os.makedirs('../data/raw', exist_ok=True)

# Save noisy version — this replaces the clean one as your working dataset
records_noisy = df_noisy.to_dict(orient='records')

with open('../data/raw/criteo_noisy.json', 'w') as f:
    json.dump(records_noisy, f, default=str)

print(f"Saved noisy dataset: {len(records_noisy):,} records")
print(f"File: data/raw/criteo_noisy.json")

Saved noisy dataset: 102,000 records
File: data/raw/criteo_noisy.json


In [6]:
#replacing clean data with updated noisy additions

from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['ecommerce_db']

# Drop the clean collection and replace with noisy version
db['criteo_raw'].drop()
collection = db['criteo_raw']

# Insert in batches
batch_size = 10000
for i in range(0, len(records_noisy), batch_size):
    batch = records_noisy[i:i+batch_size]
    collection.insert_many(batch)
    print(f"Inserted {min(i+batch_size, len(records_noisy)):,} / {len(records_noisy):,}")

total = collection.count_documents({})
print(f"\nMongoDB updated: {total:,} documents in criteo_raw")
print("Ready for ETL cleaning in m3_criteo_etl notebook")

Inserted 10,000 / 102,000
Inserted 20,000 / 102,000
Inserted 30,000 / 102,000
Inserted 40,000 / 102,000
Inserted 50,000 / 102,000
Inserted 60,000 / 102,000
Inserted 70,000 / 102,000
Inserted 80,000 / 102,000
Inserted 90,000 / 102,000
Inserted 100,000 / 102,000
Inserted 102,000 / 102,000

MongoDB updated: 102,000 documents in criteo_raw
Ready for ETL cleaning in m3_criteo_etl notebook
